In [12]:

using Statistics

# Read the original file to understand the format
original_content = read("./sparc_pf1u.dat", String)
original_lines = split(strip(original_content), '\n')

# Extract header if present
header_line = ""
data_lines = []
for line in original_lines
    line_stripped = strip(line)
    if !isempty(line_stripped) && !startswith(line_stripped, '#')
        parts = split(line_stripped)
        if isempty(header_line) && length(parts) == 4
            header_line = line_stripped
        else
            push!(data_lines, line_stripped)
        end
    end
end

# Parse the points and check number of columns
points = []
n_cols = 0
for line in data_lines
    parts = split(line)
    if n_cols == 0
        n_cols = length(parts)
    end
    if length(parts) >= 2
        try
            r = parse(Float64, parts[1])
            z = parse(Float64, parts[2])
            push!(points, [r, z])
        catch
            continue
        end
    end
end

points = hcat(points...)'

# Calculate centroid
r_center = mean(points[:, 1])
z_center = mean(points[:, 2])

# Calculate radii from center
radii = sqrt.((points[:, 1] .- r_center).^2 .+ (points[:, 2] .- z_center).^2)
angles = atan.(points[:, 2] .- z_center, points[:, 1] .- r_center)

# Bin by angle and average radius in each bin
n_bins = 32
angle_bins = range(-π, π, length=n_bins + 1)
symmetric_points = []

for i in 1:n_bins
    mask = (angles .>= angle_bins[i]) .& (angles .< angle_bins[i + 1])
    if any(mask)
        avg_r = mean(radii[mask])
        angle = (angle_bins[i] + angle_bins[i + 1]) / 2
        r = r_center + avg_r * cos(angle)
        z = z_center + avg_r * sin(angle)
        push!(symmetric_points, [r, z])
    end
end

symmetric_points = hcat(symmetric_points...)'

# Determine header format
if !isempty(header_line)
    header_parts = split(strip(header_line))
    if length(header_parts) >= 4
        ncoil = header_parts[1]
        s = header_parts[2]
        nw = header_parts[4]
        new_nsec = size(symmetric_points, 1)
        new_header = "$ncoil $s $new_nsec $nw"
    else
        new_header = "1 1 $(size(symmetric_points, 1)) 1"
    end
else
    new_header = "1 1 $(size(symmetric_points, 1)) 1"
end

# Write output file with same number of columns as original
output_path = "./sparc_pf1u_axisymmetric.dat"
open(output_path, "w") do f
    println(f, new_header)
    for i in 1:size(symmetric_points, 1)
        if n_cols == 3
            # Add third column (typically 0 for planar coils)
            println(f, "$(symmetric_points[i, 1])  $(symmetric_points[i, 2])  0.0")
        else
            println(f, "$(symmetric_points[i, 1])  $(symmetric_points[i, 2])")
        end
    end
end

println("Header: $new_header")
println("Created axisymmetric geometry with $(size(symmetric_points, 1)) points")
println("Number of columns: $n_cols")


Header: 1 1 32 1.00
Created axisymmetric geometry with 32 points
Number of columns: 3
